# ReAct Research Agent with Tavily + Firecrawl

Build a research agent that **searches the web** (Tavily) and **reads full pages** (Firecrawl),
reasoning step-by-step with lionagi's ReAct loop.

**Setup**: Add to your `.env`:
```
TAVILY_API_KEY=tvly-...
FIRECRAWL_API_KEY=fc-...
OPENAI_API_KEY=sk-...
```

## 1. Define Search & Scrape Tools

We wrap Tavily search and Firecrawl scrape as plain async functions.
lionagi auto-generates tool schemas from the signatures.

In [1]:
from lionagi.service.connections.match_endpoint import match_endpoint

_tavily = match_endpoint("tavily", "search")
_firecrawl = match_endpoint("firecrawl", "scrape")


async def web_search(
    query: str,
    max_results: int = 5,
    topic: str = "general",
) -> str:
    """Search the web using Tavily. Returns titles, URLs, and snippets.

    Args:
        query: What to search for.
        max_results: Number of results (1-20, default 5).
        topic: Search category - 'general', 'news', or 'finance'.
    """
    resp = await _tavily.call(
        {"query": query, "max_results": max_results, "topic": topic, "include_answer": True}
    )
    parts = []
    if resp.get("answer"):
        parts.append(f"Quick answer: {resp['answer']}\n")
    for r in resp.get("results", []):
        parts.append(f"- [{r['title']}]({r['url']})\n  {r.get('content', '')[:300]}")
    return "\n".join(parts) or "No results found."


async def read_page(url: str) -> str:
    """Scrape and read the full content of a web page. Returns markdown text.

    Args:
        url: The URL to read.
    """
    resp = await _firecrawl.call({"url": url, "onlyMainContent": True})
    if resp.get("success") and resp.get("data"):
        md = resp["data"].get("markdown", "")
        return md[:5000] if md else "Page loaded but no text content extracted."
    return f"Failed to read page: {resp}"


print("Tools ready: web_search, read_page")

Tools ready: web_search, read_page


## 2. Quick Tool Test

Verify both tools work before handing them to the agent.

In [2]:
results = await web_search("latest developments in AI agents 2025", max_results=3)
print(results)

Quick answer: In 2025, AI agents will deeply integrate into workflows, automate complex tasks, and provide personalized, proactive solutions. Autonomous coding AI agents will generate, test, and debug code from natural language goals. The shift from theory to infrastructure marks a significant evolution in human-AI interaction.

- [AI Agents: The Revolution Transforming Business in 2025 - Medium](https://medium.com/activated-thinker/ai-agents-the-revolution-transforming-business-in-2025-21c282977eed)
  By 2025, autonomous coding AI agents take natural language goals, generate code, write and run tests, analyze results, and autonomously debug
- [The Future of AI Agents: Top Predictions and Trends to Watch in 2026](https://www.salesforce.com/news/stories/future-of-salesforce/)
  2025 delivered enterprise AI that moved beyond simple prompts and reactive text generation into a new reality where digital agents don't just talk — they
- [AI agents 2025 recap: What happened and what to expect 

In [3]:
page = await read_page("https://github.com/lion-agi/lionagi")
print(page[:1000])

[Skip to content](https://github.com/khive-ai/lionagi#start-of-content)

You signed in with another tab or window. [Reload](https://github.com/khive-ai/lionagi) to refresh your session.You signed out in another tab or window. [Reload](https://github.com/khive-ai/lionagi) to refresh your session.You switched accounts on another tab or window. [Reload](https://github.com/khive-ai/lionagi) to refresh your session.Dismiss alert

{{ message }}

[khive-ai](https://github.com/khive-ai)/ **[lionagi](https://github.com/khive-ai/lionagi)** Public

- [Notifications](https://github.com/login?return_to=%2Fkhive-ai%2Flionagi) You must be signed in to change notification settings
- [Fork\\
73](https://github.com/login?return_to=%2Fkhive-ai%2Flionagi)
- [Star\\
384](https://github.com/login?return_to=%2Fkhive-ai%2Flionagi)


main

[**86** Branches](https://github.com/khive-ai/lionagi/branches) [**191** Tags](https://github.com/khive-ai/lionagi/tags)

[Go to Branches page](https://github.com/khive-ai/l

## 3. ReAct Research Agent

The agent can **search → read → reason → search again** in a loop.
It decides on its own when to search, when to read a full page, and when it has enough info.

In [4]:
from lionagi import Branch, iModel

researcher = Branch(
    system=(
        "You are a research assistant with web search and page reading capabilities. "
        "When answering questions:\n"
        "1. Search the web to find relevant sources\n"
        "2. Read promising pages for detailed information\n"
        "3. Synthesize findings into a clear, sourced answer\n"
        "Always cite your sources with URLs."
    ),
    tools=[web_search, read_page],
    chat_model=iModel(provider="openai", model="gpt-4.1-mini"),
)

print("Research agent ready.")

Research agent ready.


In [5]:
answer = await researcher.ReAct(
    instruct={
        "instruction": (
            "What are the key differences between Tavily and Perplexity APIs "
            "for building AI agent search tools? Compare pricing, features, "
            "and best use cases. Read at least one page for detailed info."
        ),
    },
    max_extensions=8,
    verbose=True,
)

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(answer)

### ReAct Round No.1 Analysis:

```yaml
analysis: I will first perform targeted web searches to find the official API documentation, pricing pages, or trustworthy reviews for both Tavily and Perplexity.ai. This will enable me to extract and compare exact details about their feature sets, pricing plans, and recommended use cases for AI agent search tools.
planned_actions:
  - action_type: search_exa
    description: Search for Tavily API official documentation, pricing and features
  - action_type: search_exa
    description: Search for Perplexity API official documentation, pricing and features
extension_needed: True
milestone: Collect detailed data on Tavily and Perplexity APIs before comparison
action_strategy: concurrent
reason:
  title: Need to compare Tavily and Perplexity APIs
  content: To provide an accurate comparison of Tavily and Perplexity APIs in terms of pricing, features, and use cases, I need to gather detailed information from their official or reliable sources because such info is specific and technical.
  confidence_score: None
```

### ReAct Round No.2 Analysis:

```yaml
analysis: Starting with concurrent web searches to quickly locate authoritative and detailed sources for Tavily and Perplexity APIs. After this, I plan to read relevant pages to extract key details required for comparison.
planned_actions:
  - action_type: web_search
    description: Search for Tavily API documentation and pricing details.
  - action_type: web_search
    description: Search for Perplexity API documentation and pricing details.
extension_needed: True
milestone: Retrieve relevant search results for Tavily and Perplexity API pricing and features
action_strategy: concurrent
reason:
  title: Initial Web Searches for Tavily and Perplexity APIs
  content: To compare Tavily and Perplexity APIs effectively, I need to gather up-to-date data about their pricing, feature sets, and recommended use cases. This requires finding official documentation or trusted review pages for both platforms.
  confidence_score: None
action_required: True
action_requests:
  - function: web_search
    arguments:
      query: Tavily API pricing features documentation
      max_results: 5
      topic: general
  - function: web_search
    arguments:
      query: Perplexity API pricing features documentation
      max_results: 5
      topic: general
action_responses:
  - function: web_search
    arguments:
      query: Tavily API pricing features documentation
      max_results: 5
      topic: general
    output: |
      Quick answer: Tavily API pricing is based on API credits; advanced search costs 2 credits per request. Free API credits are available without a credit card. Pricing details are on the Tavily website.
      
      - [Credits & Pricing - Tavily Docs](https://docs.tavily.com/documentation/api-credits)
        [Skip to main content](https://docs.tavily.com/documentation/api-credits#content-area). *   [Get an API key](https://app.tavily.com/). *   [Get an API key](https://app.tavily.com/). [Home](https://docs.tavily.com/welcome)[Introduction](https://docs.tavily.com/documentation/about)[API & SDKs](https:/
      - [Frequently Asked Questions - Tavily Docs](https://docs.tavily.com/faq/faq)
        [Skip to main content](https://docs.tavily.com/faq/faq#content-area). *   [Get an API key](https://app.tavily.com/). *   [Get an API key](https://app.tavily.com/). *   [API Playground](https://app.tavily.com/playground). *   [About](https://docs.tavily.com/documentation/about). *   [Quickstart](http
      - [Introduction - Tavily Docs](https://docs.tavily.com/documentation/api-reference/introduction)
        The base URL for all requests to the Tavily API is:. All Tavily endpoints are authenticated using API keys. Get your free API key. curl -X POST https://api.tavily.com/search \ curl -X  POST https://api.tavily.com/search  \ -H "Content-Type: application/json" \ -H "Content-Type: application/json"  \ 
      - [About - Tavily Docs](https://docs.tavily.com/documentation/about)
        ## ​ What is the Tavily Search Engine? Scraping doesn’t scale and requires expertise to refine, current search engine APIs don’t provide explicit information to queries but simply potential related articles (which are not always related), and are not very customziable for AI agent needs. Unlike othe
      - [Find a plan to power your AI Agents - Tavily](https://www.tavily.com/pricing)
        Image 31,000 API credits / month. Image 94,000 API credits / month. Accept All Cookies Reject All. When you visit any website, it may store or retrieve information on your browser, mostly in the form of cookies. However, blocking some types of cookies may impact your experience of the site and the s
  - function: web_search
    arguments:
      query: Perplexity API pricing features documentation
      max_results: 5
      topic: general
    output: |
      Quick answer: Perplexity API uses a pay-as-you-go credit system based on token usage, with no flat monthly fee. Pricing varies by model and search mode. Perplexity Pro subscribers receive a $5 monthly credit.
      
      - [Perplexity Enterprise Pricing - Get Started Today](https://www.perplexity.ai/enterprise/pricing)
        [](https://www.perplexity.ai/enterprise/pricing). [Customers](https://www.perplexity.ai/enterprise/customers). [Pricing](https://www.perplexity.ai/enterprise/pricing). [Get started](https://pplx.ai/pwmYPXv). [Get started](https://www.perplexity.ai/onboarding/org/create?utm_source=ent-lp-customer&ref
      - [Pricing - Perplexity](https://docs.perplexity.ai/docs/getting-started/pricing)
        The Agent API provides access to third-party models from OpenAI, Anthropic, Google, and xAI with **transparent, token-based pricing** at direct provider rates with no markup. If a model makes 3 web searches during a request, you pay model tokens + (3 × $0.005) for searches. ## [​](#search-api-pricin
      - [Home | Perplexity Help Center](https://www.perplexity.ai/help-center/en/)
        1 author5 articles](https://www.perplexity.ai/help-center/en/collections/8934263-what-is-perplexity)[Perplexity Pro and Max. Subscription Plans | Pricing | Product Tiers. 1 author8 articles](https://www.perplexity.ai/help-center/en/collections/8935108-perplexity-pro-and-max)[Perplexity Product Featu
      - [Perplexity AI Pricing: A Comprehensive Guide for 2025](https://www.photonpay.com/hk/blog/article/perplexity-ai-pricing?lang=en)
        The plans include a free version, a professional plan, and enterprise options, each with distinct premium features like access to advanced AI models, file analysis, and subscription renewals. When comparing features across perplexity ai pricing plans 2025, the differences become clear in areas like 
      - [Perplexity API Cost 2025: Full Pricing & Credit Guide - Global GPT](https://www.glbgpt.com/hub/perplexity-api-cost-2025/)
        # Perplexity API Cost 2026: Full Pricing & Credit Guide. As of late 2026, the Perplexity API operates on a strict pay-as-you-go credit system based on token usage and search depth, rather than a flat monthly subscription fee. While Perplexity Pro subscribers receive a recurring $5 monthly credit to
```

### ReAct Round No.3 Analysis:

```yaml
analysis: Reading both official API credit/pricing pages for Tavily and Perplexity will reveal detailed pricing models, credit usage conditions, API features, and possibly recommended use cases. This will enable a thorough comparison.
planned_actions:
  - action_type: read_page
    description: Read Tavily API credits and pricing documentation page for detailed pricing and features.
  - action_type: read_page
    description: Read Perplexity enterprise pricing page for detailed multi-tier pricing and API features.
extension_needed: True
milestone: Gather detailed pricing, features, and usage info from official API docs for Tavily and Perplexity
action_strategy: concurrent
reason:
  title: Next step: Read detailed official pages for Tavily and Perplexity pricing and features
  content: The search results provide overview snippets and links that mention credit-based pricing and features for both Tavily and Perplexity APIs. To accurately compare them, I need to read the full content of the top pages from their official documentation and pricing pages.
  confidence_score: None
action_required: True
action_requests:
  - function: read_page
    arguments:
      url: https://docs.tavily.com/documentation/api-credits
  - function: read_page
    arguments:
      url: https://www.perplexity.ai/enterprise/pricing
action_responses:
  - function: read_page
    arguments:
      url: https://docs.tavily.com/documentation/api-credits
    output: |
      [Skip to main content](https://docs.tavily.com/documentation/api-credits#content-area)
      
      [Tavily Docs home page![light logo](https://mintcdn.com/tavilyai/3bRbNpTHEFkRcfMH/logo/light.svg?fit=max&auto=format&n=3bRbNpTHEFkRcfMH&q=85&s=7df00865e87a791d3152072e23161d63)![dark logo](https://mintcdn.com/tavilyai/3bRbNpTHEFkRcfMH/logo/dark.svg?fit=max&auto=format&n=3bRbNpTHEFkRcfMH&q=85&s=bbd8aed61d89c892eafdcb7d9473f104)](https://tavily.com/)
      
      Search...
      
      Ctrl KAsk AI
      
      Search...
      
      Navigation
      
      Getting Started
      
      Credits & Pricing
      
      [Home](https://docs.tavily.com/welcome) [Introduction](https://docs.tavily.com/documentation/about) [API & SDKs](https://docs.tavily.com/documentation/api-reference/introduction) [Ecosystem](https://docs.tavily.com/documentation/mcp) [Examples](https://docs.tavily.com/examples/use-cases/chat) [Changelog](https://docs.tavily.com/changelog) [Help](https://docs.tavily.com/documentation/help)
      
      - [API Playground](https://app.tavily.com/playground)
      - [Community](https://discord.gg/TPu2gkaWp2)
      - [Blog](https://tavily.com/blog)
      
      ##### Getting Started
      
      - [About](https://docs.tavily.com/documentation/about)
      - [Quickstart](https://docs.tavily.com/documentation/quickstart)
      - [Credits & Pricing](https://docs.tavily.com/documentation/api-credits)
      - [Rate Limits](https://docs.tavily.com/documentation/rate-limits)
      
      ##### FAQ
      
      - [Frequently Asked Questions](https://docs.tavily.com/faq/faq)
      
      On this page
      
      - [Free API Credits](https://docs.tavily.com/documentation/api-credits#free-api-credits)
      - [Pricing Overview](https://docs.tavily.com/documentation/api-credits#pricing-overview)
      - [API Credits Costs](https://docs.tavily.com/documentation/api-credits#api-credits-costs)
      - [Tavily Search](https://docs.tavily.com/documentation/api-credits#tavily-search)
      - [Tavily Extract](https://docs.tavily.com/documentation/api-credits#tavily-extract)
      - [Tavily Map](https://docs.tavily.com/documentation/api-credits#tavily-map)
      - [Tavily Crawl](https://docs.tavily.com/documentation/api-credits#tavily-crawl)
      - [Tavily Research](https://docs.tavily.com/documentation/api-credits#tavily-research)
      
      ## [​](https://docs.tavily.com/documentation/api-credits\#free-api-credits)  Free API Credits
      
      [**Get your free API key** \\
      \\
      You get 1,000 free API Credits every month. **No credit card required.**](https://app.tavily.com/)
      
      ## [​](https://docs.tavily.com/documentation/api-credits\#pricing-overview)  Pricing Overview
      
      Tavily operates on a simple, credit-based model:
      
      - **Free**: 1,000 credits/month
      - **Pay-as-you-go**: $0.008 per credit (allows you to be charged per credit once your plan’s credit limit is reached).
      - **Monthly plans**: $0.0075 - $0.005 per credit
      - **Enterprise**: Custom pricing and volume
      
      | **Plan** | **Credits per month** | **Monthly price** | **Price per credit** |
      | --- | --- | --- | --- |
      | **Researcher** | 1,000 | Free | - |
      | **Project** | 4,000 | $30 | $0.0075 |
      | **Bootstrap** | 15,000 | $100 | $0.0067 |
      | **Startup** | 38,000 | $220 | $0.0058 |
      | **Growth** | 100,000 | $500 | $0.005 |
      | **Pay as you go** | Per usage | $0.008 / Credit | $0.008 |
      | **Enterprise** | Custom | Custom | Custom |
      
      Head to [billing](https://app.tavily.com/billing) to explore our different options and manage your plan.
      
      ## [​](https://docs.tavily.com/documentation/api-credits\#api-credits-costs)  API Credits Costs
      
      ### [​](https://docs.tavily.com/documentation/api-credits\#tavily-search)  Tavily Search
      
      Your [search depth](https://docs.tavily.com/api-reference/endpoint/search#body-search-depth) determines the cost of your request.
      
      - **Basic Search (`basic`):**
      Each request costs **1 API credit**.
      - **Advanced Search (`advanced`):**
      Each request costs **2 API credits**.
      
      ### [​](https://docs.tavily.com/documentation/api-credits\#tavily-extract)  Tavily Extract
      
      The number of successful URL extractions and your [extraction depth](https://docs.tavily.com/api-reference/endpoint/extract#body-extract-depth) determines the cost of your request. You never get charged if a URL extraction fails.
      
      - **Basic Extract (`basic`):**
      Every 5 successful URL extractions cost **1 API credit**
      - **Advanced Extract (`advanced`):**
      Every 5 successful URL extractions cost **2 API credits**
      
      ### [​](https://docs.tavily.com/documentation/api-credits\#tavily-map)  Tavily Map
      
      The number of pages mapped and whether or not natural-language [instructions](https://docs.tavily.com/documentation/api-reference/endpoint/map#instructions) are specified determines the cost of your request. You never get charged if a map request fails.
      
      - **Regular Mapping:**
      Every 10 successful pages returned cost **1 API credit**
      - **Map with (`instructions`):**
      Every 10 successful pages returned cost **2 API credits**
      
      ### [​](https://docs.tavily.com/documentation/api-credits\#tavily-crawl)  Tavily Crawl
      
      Tavily Crawl combines both mapping and extraction operations, so the cost is the sum of both:
      
      - **Crawl Cost = Mapping Cost + Extraction Cost**
      
      For example:
      
      - If you crawl 10 pages
  - function: read_page
    arguments:
      url: https://www.perplexity.ai/enterprise/pricing
    output: |
      # Give your team their time back
      
      ### Whether you’re a small business or a large enterprise, Perplexity Enterprise enables your team to think strategically rather than search endlessly.
      
      [Get started](https://www.perplexity.ai/onboarding/org/create?utm_source=ent-lp-customer&ref=perplexity.ai%2Fenterprise&utm_medium=website)
      
      [Request a demo](https://forms.default.com/316402)
      
      For personal, non-commercial use and subject to our [policies](https://www.perplexity.ai/hub/legal/terms-of-service). For fast answers powered by the latest AI models.
      
      ## $17 / month
      
      ## when billed annually
      
      [Learn more](https://www.perplexity.ai/pro)
      
      Key features:
      
      ## Access to the latest Al models, post-trained for higher accuracy
      
      ## Select between recent models of GPT, Claude, Gemini, and more
      
      ## Better for complex questions and building reports, documents, and apps
      
      ## Deeper sourcing from Perplexity's index, including proprietary financial and scientific data from PitchBook, Wiley, and more
      
      ## Usage limits best for most users
      
      Designed for teams and organizations streamlining everyday research with accurate answers, securely.
      
      ## $34 / month per seat
      
      ## when billed annually
      
      [Get started](https://pplx.ai/EhPzIXW)
      
      All the features of Perplexity Pro, and:
      
      ## Guaranteed no training on your data
      
      ## Access to the latest Al models, post-trained for higher accuracy
      
      ## Search across the web, team files, and work apps
      
      ## Premium citations from PitchBook, Statista and more
      
      ## Single Sign On or SCIM provisioning
      
      ## User management and permissioning
      
      ## 2x file uploads
      
      ## Dedicated Enterprise support
      
      ## SOC 2 Type II, HIPAA, GDPR, PCI DSS compliant
      
      Specifically built for users tackling complex challenges with unlimited queries and the best models.
      
      ## $271 / month per seat
      
      ## when billed annually
      
      [Get started](https://pplx.ai/EhPzIXW)
      
      All the features of Enterprise Pro, and:
      
      ## Guaranteed no training on your data
      
      ## Get answers from the most advanced Al reasoning models (e.g., Claude Opus)
      
      ## Run deep research at any scale
      
      ## Work with larger datasets and files
      
      ## Greater file upload limits
      
      ## Compare responses across multiple AI models
      
      ## Data retention configurability, audit logs, and team insights
      
      ## SOC 2 Type II, HIPAA, GDPR, PCI DSS compliant
      
      ## Features
      
      Pro
      
      [Learn more](http://perplexity.ai/pro/payment)
      
      Enterprise Pro
      
      [Get started](https://pplx.ai/EhPzIXW)
      
      Enterprise Max
      
      [Get started](https://pplx.ai/EhPzIXW)
      
      Platform
      
      Pro queries
      
      (multi-step reasoning searches and additional sources)
      
      Up to 200 per week
      
      2x Pro queries
      
      20x Pro queries
      
      Deep Research queries
      
      (in-depth research reports on any topic)
      
      Up to 20 per month
      
      2.5x Pro queries
      
      25x Pro queries
      
      Asset generation
      
      Up to 25 per month
      
      2x Pro assets
      
      20x Pro assets
      
      Video generation
      
      3 videos per month
      
      5 videos per month
      
      15 high-quality videos with audio a month
      
      Collaborate in private Spaces with teammates
      
      5 collaborators per Space
      
      Unlimited collaborators
      
      Unlimited collaborators
      
      Ability to choose different models to search on
      
      Select your preferred AI model
      
      Select your preferred AI model
      
      Access to additional advanced AI models and multi-model research mode
      
      Get answers from files
      
      (files must be <50mb)
      
      50 uploads per week
      
      2x Pro uploads
      
      20x Pro uploads
      
      Organizational file repository
      
      Configurable controls around file sharing
      
      Configurable controls around file sharing
      
      Search through file apps
      
      Attach files from Google Drive and Dropbox
      
      [Sync files from Google Drive, Sharepoint, and other file apps](https://www.perplexity.ai/enterprise/file-app-connectors)
      
      [Sync files from Google Drive, Sharepoint, and other file apps](https://www.perplexity.ai/enterprise/file-app-connectors)
      
      Search and write to apps
      
      [Get answers and update your apps like Salesforce, Hubspot, Slack, and 100+ others](https://www.perplexity.ai/enterprise/app-connectors)
      
      [Get answers and update your apps like Salesforce, Hubspot, Slack, and 100+ others](https://www.perplexity.ai/enterprise/app-connectors)
      
      Memory
      
      Configurable memory permissions for admins and users
      
      Configurable memory permissions for admins and users
      
      Model council
      
      Compare your answers with all the top models in one answer
      
      Comet
      
      Comet Assistant
      
      Assistant in Comet
      
      Assistant in Comet
      
      Advanced models powering Comet assistant
      
      Comet Agent
      
      40 queries per month
      
      2x Pro queries
      
      20x Pro queries
      
      Computer
      
      Computer credits†
      
      15,000 credits per month
      
      † Available for paying subscribers.
      
      Admin Controls + Security
      
      Single sign on + SCIM\*
      
      Integrate with your identity provider for SSO + SCIM
      
      Integrate with your identity provider for SSO + SCIM
      
      User management
      
      Admins can add, remove, and set permissions across team members
      
      Admins can add, remove, and set permissions across team members
      
      Insights dashboard\*
      
      Analyze your usage trends and track adoption
      
      Analyze your usage trends and track adoption
      
      Data retention options\*
      
      Ability to auto-delete files within 1 day
      
      Ability to auto-delete fil
```

### ReAct Round No.4 Analysis:

```yaml
analysis: Based on official documentation: Tavily is cost-effective for scalable search requests with clear credit costs per feature (search, extract, map, crawl) and free monthly credits, making it suitable for developers needing flexible pay-as-you-go or volume discounted credit usage. Perplexity provides a structured SaaS model with seats and tiered plans including advanced AI use, multi-model research, extensive integrations, collaboration, and enterprise security, suited for teams and enterprises requiring built-in platform features beyond just API calls.
planned_actions:
  - action_type: None
    description: Summarize and contrast key differences of Tavily and Perplexity APIs including pricing, features, and best use cases.
extension_needed: False
milestone: Compare Tavily and Perplexity APIs and finalize summary
action_strategy: concurrent
reason:
  title: Detailed pricing and feature reading completed
  content: I have analyzed the detailed pricing and feature information from both Tavily and Perplexity official pages. Tavily uses a credit-based system with free and tiered monthly plans, charging from $0.008 to $0.005 per credit depending on volume, with specifics on costs per search depth and usage. Perplexity offers Pro, Enterprise Pro, and Enterprise Max plans with monthly seat-based pricing ($17 to $271 per month billed annually) providing tiered access to advanced AI models, multi-source search, enterprise-grade features like file uploads, app integrations, admin controls, and compliance certifications. Their pricing is more seat-based with included query limits, while Tavily is primarily pay-per-credit with volume discounts. I now have the necessary data to compare pricing, features, and best use cases.
  confidence_score: None
action_required: False
action_requests:
action_responses: None
```

### ReAct Final Answer:

```yaml
answer: |
  The key differences between Tavily and Perplexity APIs for building AI agent search tools lie in their pricing models, feature sets, and ideal use cases:
  
  1. Pricing:
  - Tavily uses a credit-based pricing system with a free tier of 1,000 API credits per month and pay-as-you-go pricing at $0.008 per credit. Monthly plans offer volume discounts, ranging from $0.0075 to $0.005 per credit depending on monthly credit bundles. Credit consumption varies by operation type, e.g., basic search costs 1 credit per request, advanced search 2 credits.
  - Perplexity offers subscription-based pricing with seat licenses billed monthly or annually. Plans range from Perplexity Pro at $17/month per seat to Enterprise Max at $271/month per seat, including query limits, access to advanced AI models, and enhanced collaboration features. Pricing reflects enterprise features beyond pure API usage.
  
  2. Features:
  - Tavily focuses on flexible AI-powered search APIs optimized for AI agents with explicit control over search depth, extraction, mapping, and crawling operations. It provides granular credit costs for these API functions and supports free API keys without credit cards.
  - Perplexity provides an AI research platform with multi-model access (GPT, Claude, Gemini, etc.), multi-source web and file search, app integrations, collaboration tools, user management, compliance certifications (SOC 2, HIPAA, GDPR), and advanced enterprise controls. It targets teams needing a full research and agent-building environment including data sync and model comparisons.
  
  3. Best Use Cases:
  - Tavily is best suited for developers and organizations requiring scalable, customizable search APIs with a cost-effective pay-as-you-go or volume discount credit system. It is ideal when you want granular API usage control and are building custom AI agents focused on advanced search and data extraction.
  - Perplexity is better for teams and enterprises wanting a managed SaaS platform with built-in AI models, collaboration, security, and integration with corporate data sources and applications. It serves use cases needing heavy multi-model research, compliance, and administrative controls alongside agent AI.
  
  In summary, Tavily is a credit-based, developer-friendly API platform emphasizing flexible search-related operations and cost control, while Perplexity provides a higher-tier SaaS solution with advanced AI research capabilities and enterprise features suited for teams and organizations.
  
  Sources:
  - Tavily Docs API Credits & Pricing: https://docs.tavily.com/documentation/api-credits
  - Perplexity Enterprise Pricing: https://www.perplexity.ai/enterprise/pricing
```


FINAL ANSWER:
The key differences between Tavily and Perplexity APIs for building AI agent search tools lie in their pricing models, feature sets, and ideal use cases:

1. Pricing:
- Tavily uses a credit-based pricing system with a free tier of 1,000 API credits per month and pay-as-you-go pricing at $0.008 per credit. Monthly plans offer volume discounts, ranging from $0.0075 to $0.005 per credit depending on monthly credit bundles. Credit consumption varies by operation type, e.g., basic search costs 1 credit per request, advanced search 2 credits.
- Perplexity offers subscription-based pricing with seat licenses billed monthly or annually. Plans range from Perplexity Pro at $17/month per seat to Enterprise Max at $271/month per seat, including query limits, access to advanced AI models, and enhanced collaboration features. Pricing reflects enterprise features beyond pure API usage.

2. Features:
- Tavily focuses on flexible AI-powered search APIs optimized for AI agents with explic

## 4. Structured Output with ReAct

Get research results as a Pydantic model instead of free text.

In [6]:
from lionagi import BaseModel, Field


class Source(BaseModel):
    title: str
    url: str


class ResearchReport(BaseModel):
    title: str = Field(description="Report title")
    summary: str = Field(description="2-3 sentence executive summary")
    key_findings: list[str] = Field(description="Main findings as bullet points")
    sources: list[Source] = Field(description="Sources used")


agent2 = Branch(
    system=(
        "You are a research analyst. Search the web, read pages for detail, "
        "and produce structured research reports with citations."
    ),
    tools=[web_search, read_page],
    chat_model=iModel(provider="openai", model="gpt-4.1-mini"),
)

report = await agent2.ReAct(
    instruct={
        "instruction": (
            "Research the current state of open-source AI agent frameworks in 2025. "
            "Which are the most popular? What are the key trends?"
        ),
    },
    max_extensions=8,
    verbose=True,
    response_format=ResearchReport,
)

print("\n" + "=" * 60)
print(f"Title: {report.title}")
print(f"\nSummary: {report.summary}")
print("\nKey Findings:")
for f in report.key_findings:
    print(f"  - {f}")
print("\nSources:")
for s in report.sources:
    print(f"  - [{s.title}]({s.url})")

### ReAct Round No.1 Analysis:

```yaml
analysis: The landscape of AI agent frameworks is rapidly evolving, especially in 2025, with increasing interest in autonomous AI systems and multi-agent coordination. Popular frameworks typically have strong community support, frequent updates, ease of integration with various LLMs, and versatile use cases spanning automation, personal assistants, research, and AI orchestration. Key trends likely include modularity, plug-and-play LLM support, integration with cloud and edge AI infrastructure, improved safety and robustness features, and enhanced collaboration tools. To validate and elaborate, targeted web searches will be conducted next to identify leading open-source AI agent frameworks and summarize key trends.
planned_actions:
  - action_type: search_exa
    description: Search for recent reviews, developer discussions, or rankings of open-source AI agent frameworks in 2025, focusing on popularity and community adoption.
extension_needed: True
milestone: Identify major open-source AI agent frameworks in 2025 and summarize their popularity and trends.
action_strategy: sequential
reason:
  title: Initial planning and overview
  content: To produce a rigorous research report on the current state of open-source AI agent frameworks in 2025, including their popularity and key trends, the best approach is to gather recent data and authoritative sources. This includes examining developer activity on platforms like GitHub, key mentions in AI and developer communities, recent articles, and vendor or community announcements. This first step will involve performing web searches to identify top open-source AI agent frameworks as of 2025 and noting any emerging trends.
  confidence_score: 0.9
```

### ReAct Round No.2 Analysis:

```yaml
analysis: This is the critical next step to gather updated, concrete information on which AI agent frameworks are leading in 2025. The search results should reveal names of popular frameworks, their features, and community activity, as well as emerging trends such as multi-agent coordination, modular LLM integration, or safety advancements.
planned_actions:
  - action_type: web_search
    description: Search for recent reviews and rankings of the top open-source AI agent frameworks in 2025 with a focus on popularity and community adoption.
extension_needed: True
milestone: Identify major open-source AI agent frameworks in 2025 and summarize their popularity and trends
action_strategy: concurrent
reason:
  title: Search for recent reviews and rankings of open-source AI agent frameworks in 2025
  content: To identify the most popular open-source AI agent frameworks in 2025, I will perform a targeted web search focusing on developer discussions, articles, and repository rankings from platforms such as GitHub and tech forums. Key search queries will include terms like 'best open source AI agent frameworks 2025', 'top AI agent frameworks open source', and 'AI agent frameworks popularity 2025'. This will help gather fresh insights and confirm current trends.
  confidence_score: 0.9
action_required: None
action_requests: None
action_responses: None
```

### ReAct Round No.3 Analysis:

```yaml
analysis: This search should yield a list of frameworks that are considered most popular in 2025 with insights into their community engagement, feature sets, and key trends such as modularity, multi-agent coordination, or integrations with large language models. This will form the basis for a detailed report.
planned_actions:
  - action_type: web_search
    description: Search for recent reviews and rankings of open source AI agent frameworks in 2025, focusing on popularity, community feedback, and trends.
extension_needed: True
milestone: Gather and validate information on most popular open source AI agent frameworks in 2025 and emerging trends.
action_strategy: sequential
reason:
  title: Perform a web search for most popular open-source AI agent frameworks in 2025
  content: To identify the leading open-source AI agent frameworks and key trends in 2025, I will search for recent summaries, reviews, and rankings published in developer platforms, AI news sites, and GitHub. This will help capture community preferences, star counts, and new emerging capabilities. The search will use queries like 'best open source AI agent frameworks 2025', 'top open source AI agents', and 'popular AI agent frameworks 2025'.
  confidence_score: 0.95
action_required: True
action_requests:
  - function: web_search
    arguments:
      query: best open source AI agent frameworks 2025
      max_results: 7
      topic: general
action_responses:
  - function: web_search
    arguments:
      query: best open source AI agent frameworks 2025
      max_results: 7
      topic: general
    output: |
      Quick answer: In 2025, LangChain, LangGraph, and AutoGen are top open-source AI agent frameworks. They offer flexibility and scalability for building intelligent agents. CrewAI and n8n also stand out for multi-agent systems and automation workflows.
      
      - [The Best Open Source Frameworks For Building AI Agents in 2026](https://www.firecrawl.dev/blog/best-open-source-agent-frameworks)
        The Best Open Source Frameworks For Building AI Agents in 2026 image. The seven frameworks we selected show real value based on these criteria and provide solid options for developers building AI agents in 2026. ## Top 7 open source frameworks to build AI agents. CrewAI framework interface showing c
      - [The Complete Guide to Choosing an AI Agent Framework in 2025](https://www.langflow.org/blog/the-complete-guide-to-choosing-an-ai-agent-framework-in-2025)
        This is a developer-oriented, technical, and balanced review of Langflow, n8n, OpenAI's new AgentKit, LangChain (+ LangGraph), CrewAI, and AutoGPT—plus a decision matrix with 10 factors, use case recommendations, and honest pros/cons all oriented to helping you and your team choose the right tool fo
      - [Top 10 Open-Source AI Frameworks You Should Try (2025 Edition)](https://ai.plainenglish.io/top-10-open-source-ai-frameworks-you-should-try-2025-edition-f5a09d3700f6)
        Artificial Intelligence (AI) is no longer confined to research labs — it’s now powering apps, chatbots, and even the tools we use daily. But behind all that magic are **AI frameworks** — the engines that make it all possible. Let’s explore the **Top 10 Open Source AI Frameworks** you should try in 2
      - [Top AI Agent Frameworks in 2025 - Codecademy](https://www.codecademy.com/article/top-ai-agent-frameworks-in-2025)
        # Top AI Agent Frameworks in 2025. These systems, called AI agents, rely on specialized frameworks that handle memory, orchestration, and tool use. ## What are AI agent frameworks? ## Top AI agent frameworks in 2025. It offers a modular approach to building AI agents, allowing developers to chain la
      - [Top 10 Open-Source AI Agent Frameworks to Know in 2025 - ODSC](https://odsc.medium.com/top-10-open-source-ai-agent-frameworks-to-know-in-2025-c739854ec859)
        # Top 10 Open-Source AI Agent Frameworks to Know in 2025. As AI agents continue to evolve from research concepts into production-ready solutions, open-source frameworks are playing a pivotal role in accelerating adoption. Whether you’re building autonomous systems, LLM-powered applications, or orche
      - [What are the most reliable AI agent frameworks in 2025? - Reddit](https://www.reddit.com/r/aiagents/comments/1pc9r13/what_are_the_most_reliable_ai_agent_frameworks_in/)
        1. LangChain: Still the most flexible if you can code. · 2. GraphBit: This one surprised me. · 3. LangGraph: · 4. AutoGPT: · 5. Zapier / Make: · 6.
      - [Top 11 Open-Source Autonomous Agents & Frameworks in 2025](https://cline.ghost.io/top-11-open-source-autonomous-agents-frameworks-in-2025/)
        Discover the Top 11 Open-Source Autonomous coding agents and frameworks of 2025, including Cline and other top AI tools. Cline appears at #1 because it’s built specifically for day-to-day coding with local control, editor-native workflows, and model choice—while many alternatives are either general-
```

### ReAct Round No.4 Analysis:

```yaml
analysis: Reading these selected authoritative, recent comprehensive guides will allow me to consolidate data on framework features, popularity, developer support, common use cases, and key trends, producing a detailed report for 2025.
planned_actions:
  - action_type: read_page
    description: Read 'The Complete Guide to Choosing an AI Agent Framework in 2025' from langflow.org for detailed framework comparison.
  - action_type: read_page
    description: Read 'The Best Open Source Frameworks For Building AI Agents in 2026' from firecrawl.dev to verify popularity and trends.
extension_needed: True
milestone: Thoroughly review and synthesize detailed content from top 2 comprehensive framework guides to report popularity and key trends.
action_strategy: concurrent
reason:
  title: Select top sources for in-depth reading about popular open-source AI agent frameworks in 2025
  content: The web search results identify several top open-source AI agent frameworks such as LangChain, LangGraph, AutoGen, CrewAI, n8n, AutoGPT, and Cline. There are multiple useful articles offering detailed comparisons and pros/cons, e.g., the guide on langflow.org, firecrawl.dev article, and the ODSC Medium piece. To provide a comprehensive structured report on popularity and trends, I will select two high-quality resources for full reading: the 'Complete Guide to Choosing an AI Agent Framework in 2025' from langflow.org and the 'Top 7 Open Source Frameworks for 2026' from firecrawl.dev. These provide updated reviews and comparative analysis essential for drawing conclusions.
  confidence_score: 0.95
action_required: True
action_requests:
  - function: read_page
    arguments:
      url: https://www.langflow.org/blog/the-complete-guide-to-choosing-an-ai-agent-framework-in-2025
  - function: read_page
    arguments:
      url: https://www.firecrawl.dev/blog/best-open-source-agent-frameworks
action_responses:
  - function: read_page
    arguments:
      url: https://www.langflow.org/blog/the-complete-guide-to-choosing-an-ai-agent-framework-in-2025
    output: |
      ![gradient](https://www.langflow.org/images/Gradient.png?w=16&q=75)
      
      Choosing an AI agent framework in 2025 is less about picking the “best” tool (if such a thing exists) and more about aligning trade-offs with your team’s constraints: the systems you need to integrate, the governance you must meet, how quickly you need working value, and how deeply you want to customize.
      
      The right choice depends on your use case and stage, and you’ll typically need to choose from:
      
      - rapid prototyping vs. production operations,
      
      - visual builder vs. code-first,
      
      - single-agent vs. multi-agent, and
      
      - cloud-managed vs. self-hosted
      
      
      This is a developer-oriented, technical, and balanced review of Langflow, n8n, OpenAI's new AgentKit, LangChain (+ LangGraph), CrewAI, and AutoGPT—plus a decision matrix with 10 factors, use case recommendations, and honest pros/cons all oriented to helping you and your team choose the right tool for your specific use case.
      
      ## What does “agent frameworks” actually mean in 2025?
      
      An AI agent framework in 2025 typically spans several layers:
      
      - **Orchestration model**: some variety of directed graphs, conversation loops, role-based crews, or function-call workflows.
      
      - **Tooling and connectors**: we typically see components for web/file search, code execution, vector stores, SaaS connectors, OpenAPI ingestion.
      
      - **Memory and state**: conversation, episodic, or long-term memory; per-session or cross-session stores.
      
      - **Evaluations and guardrails**: like OpenAI’s AgentKit, frameworks usually come with datasets, trace grading, safety filters, and runtime validation.
      
      - **Visual builder vs. code-first APIs**: some are drag-and-drop canvases, others (like Autogen) are more code-oriented.
      
      - **Deployment and governance**: how do you deploy these? Do you self-host? How do you process audit logs, Role Based Access Control (RBAC), telemetry, and observability?
      
      - **Multi-agent architectures**: team orchestration patterns, hand-offs, human-in-the-loop primitives, and agent-to-agent protocols.
      
      
      Common pressure points here typically include developer experience, production-readiness, connectivity to enterprise systems, along with model-agnosticism, and cloud/on-premise parity requirements. Let's explore each of our options next.
      
      ## Langflow
      
      Langflow provides a node-based visual editor that composes LLMs, retrieval, and agent components. It supports multi-agent orchestration, sessionized API calls, streaming via SSE. In Langflow, components in the visual builder are just Python code and can be customized and extended as needed with code.
      
      Langflow also supports deployment as a REST API or MCP (Model Context Protocol) server. It is also an MCP client and can consume other MCP servers. All Langflow flows are JSON and can be exported, imported, and shared as such.
      
      ### Tradeoffs
      
      - Strengths
        - Visual-first, open source software (OSS), quick to prototype. Flows exportable as JSON, deployable as API, or embeddable widget; streaming and session context supported.
      
        - Custom components are straightforward classes with typed inputs/outputs; easy to integrate external services.
      
        - Full MCP support as a client and server makes Langflow flows callable tools; good for multi-tool and agent ecosystems.
      
        - Works with any models across the ecosystem from OpenAI, Anthropic, HuggingFace, and [Ollama for local-only open source models](https://www.langflow.org/blog/local-ai-using-ollama-with-agents) deployed in zero-trust enterprise architecture.
      
        - Good pairing with observability stacks (LangSmith, Langfuse).
      - Weaknesses
        - none ( _just kidding_)
      
        - Not a cloud solution: often presents complexity around hosting decisions around where to host it and how. ( [We have a blog post for that](https://www.langflow.org/blog/how-to-host-langflow))
      
        - Governance and enterprise controls exist via integrations, but you may need to assemble your own stack (auth, RBAC, audit) vs. buying a platform.
      
      **Best fit**: teams who want a visual OSS workbench to iterate on agents, inject custom Python nodes, and serve flows as APIs/MCP tools without committing to a heavy platform—all while self-hosting with peak privacy despite the complexity.
      
      ## n8n
      
      n8n is a general-purpose automation engine that now includes AI nodes and patterns for agentic workflows, including multi-agent teams and agent-to-agent delegation. It shines in orchestration primitives (retries, branches, schedules, webhooks), run logs, and a large integration catalog. Recent updates enable building multi-agent coordination patterns natively.
      
      ### Tradeoffs
      
      - Strengths
        - n8n is a mature orchestrator with run history, error handling, schedules, and hundreds of connectors; ideal glue for agents and SaaS.
      
        - Massive ecosystem of integrations and connectors.
      
        - Multi-agent patterns are supported via AI nodes, logic nodes, and agent-to-agent workflows.
      
        - It supports self-hosting like Langflow, but also offers a cloud-based SaaS solution.
      - Weaknesses
        - Not a
  - function: read_page
    arguments:
      url: https://www.firecrawl.dev/blog/best-open-source-agent-frameworks
    output: |
      Introducing Browser Sandbox - Give your agents a secure, fully managed browser environment [Read more →](https://docs.firecrawl.dev/features/browser)
      
      //
      
      Get started
      
      //
      
      ### Ready to build?
      
      Start getting Web Data for free and scale seamlessly as your project expands.No credit card needed.
      
      [Start for free](https://www.firecrawl.dev/signin) [See our plans](https://www.firecrawl.dev/pricing)
      
      #### Table of Contents
      
      [Blog](https://www.firecrawl.dev/blog)
      
      The Best Open Source Frameworks For Building AI Agents in 2026
      
      ![placeholder](https://www.firecrawl.dev/_next/image?url=%2Fblog%2Fauthors%2Fbex.jpg&w=48&q=75&dpl=dpl_3dHijKsCmAWQEV5sWrGkZXUyuPVW)Bex Tuychiev
      
      Feb 24, 2026(updated)
      
      ![The Best Open Source Frameworks For Building AI Agents in 2026 image](https://www.firecrawl.dev/_next/image?url=%2Fimages%2Fblog%2Fagent-frameworks%2Fagent-frameworks.webp&w=3840&q=75&dpl=dpl_3dHijKsCmAWQEV5sWrGkZXUyuPVW)
      
      ## TLDR
      
      - Seven open source frameworks compared: LangGraph, OpenAI Agents SDK, AutoGen, CrewAI, Google ADK, Dify, and Mastra
      - LangGraph leads in enterprise adoption (34.5M monthly downloads), Dify leads in GitHub stars (129.8k)
      - Firecrawl's /agent endpoint handles web data collection for any framework
      - 10 best practices from Anthropic, OpenAI, and McKinsey for deploying agents in production
      - Comparison table with stars, downloads, features, and best-fit use cases for each framework
      
      * * *
      
      The hype around AI agents is very real and well-deserved. According to [Markets And Markets](https://www.marketsandmarkets.com/Market-Reports/ai-agents-market-15761548.html), the global agent market reached $7.84 billion in 2025 and is projected to hit $52.62 billion by 2030, with a compound annual growth rate (CAGR) of 46.3%. [Gartner predicts](https://www.gartner.com/en/newsroom/press-releases/2025-08-26-gartner-predicts-40-percent-of-enterprise-apps-will-feature-task-specific-ai-agents-by-2026-up-from-less-than-5-percent-in-2025) that 40% of enterprise applications will feature task-specific AI agents by end of 2026, up from less than 5% in 2025. This rapid growth is fueled by stronger and stronger foundational LLMs that seem to come out every week now.
      
      For this reason, choosing appropriate frameworks to build AI agents has become essential if developers want to integrate existing infrastructure with language models. In this article, we will examine the most powerful and widely adopted open source frameworks to build agents in 2026, analyzing their technical features, implementation requirements and optimal use cases.
      
      **Enhance Your Agents:** Combine agent frameworks with [RAG systems](https://www.firecrawl.dev/blog/best-open-source-rag-frameworks) for knowledge-augmented responses. Use [web scraping libraries](https://www.firecrawl.dev/blog/best-open-source-web-scraping-libraries) and [browser automation tools](https://www.firecrawl.dev/blog/browser-automation-tools-comparison) for data collection.
      
      **Related Guides:** For converting websites into agents, see our [website to agent tutorial](https://www.firecrawl.dev/blog/website-to-agent-with-firecrawl-openai) and [website to LangGraph agent guide](https://www.firecrawl.dev/blog/website-to-langgraph-agent). For OpenAI-specific implementations, check [OpenAI agent builders and Firecrawl](https://www.firecrawl.dev/blog/openai-agent-builders-and-firecrawl).
      
      ## Our evaluation methodology to filter agent frameworks
      
      To find the best agent frameworks, we used clear metrics and practical requirements. We gathered data from GitHub, PyPI downloads, documentation, and industry articles, looking for frameworks that can handle reasoning tasks, work together as multiple agents, and [use tools effectively](https://www.firecrawl.dev/blog/agent-tools).
      
      We evaluated each framework based on:
      
      - **GitHub metrics**: Stars, active contributors, and regular updates
      - **Adoption**: Monthly download numbers
      - **Technical features**: Ability to reason, collaborate between agents, and use tools
      - **Documentation**: Clear guides and tutorials for developers
      - **Real-world use**: Proven cases in actual production environments
      - **Industry use**: How they work across different sectors like finance and customer service
      - **Backing organizations**: Support from established companies for ongoing development
      
      The seven frameworks we selected show real value based on these criteria and provide solid options for developers building AI agents in 2026.
      
      ## Firecrawl: The best web data collection agent
      
      Before we get into the list, Firecrawl deserves its own mention. What started as a web scraping API has grown into a full data collection platform for agents. The [/agent endpoint](https://docs.firecrawl.dev/features/agent) handles the hardest part of agent development: getting structured data from the web without writing scraping scripts. You describe what you want in plain text, optionally pass a Pydantic schema, and the agent searches, navigates, and returns structured results.
      
      Firecrawl's ag
```

### ReAct Round No.5 Analysis:

```yaml
analysis: |
  The combined evidence from recent reviews shows that as of 2025, LangChain, LangGraph, AutoGen, CrewAI, n8n, and AutoGPT are the most popular open-source AI agent frameworks, each with different strengths and trade-offs. Popularity is validated by metrics like monthly downloads (LangGraph leads with 34.5M), GitHub stars (Dify leads with 129.8K), and enterprise adoption. Key trends are clear: frameworks support modular orchestration models, multi-agent architectures, integrated tooling, deploy flexibility (cloud and on-prem), focus on memory/state management, and safety/guardrails. Visual vs. code-first usability is a major consideration. Enterprise governance and open model support also matter.
  
  This comprehensive, multi-source analysis enables me to now finalize the research report with details on frameworks, popularity, strengths, weaknesses, and emerging trends in 2025 open-source AI agent frameworks.
planned_actions:
  - action_type: none
    description: Synthesize and write structured research report based on gathered data and analysis.
extension_needed: False
milestone: Finalize structured research report on open-source AI agent frameworks in 2025 detailing popularity and key trends.
action_strategy: sequential
reason:
  title: Synthesize insights from detailed framework reviews
  content: |
    I have reviewed detailed descriptions of top open-source AI agent frameworks from two authoritative sources. The langflow.org guide emphasizes Langflow, n8n, OpenAI AgentKit, LangChain (+ LangGraph), CrewAI, and AutoGPT, highlighting their trade-offs around visual vs. code-first approaches, multi-agent support, deployment options, and governance. Firecrawl.dev provides a data-driven comparison of seven leading frameworks (LangGraph, OpenAI Agents SDK, AutoGen, CrewAI, Google ADK, Dify, Mastra) emphasizing adoption metrics, GitHub stars, features, and use cases. Key trends include modular orchestration, multi-agent collaboration, enterprise integrations, and memory/state management. Popularity is led by LangChain and LangGraph, with AutoGen and CrewAI also notable, plus n8n as a general-purpose automation orchestrator. 
    
    Next, I will extract and categorize detailed popularity measures, feature comparisons, and summarize key trends to consolidate into a structured research report.
  confidence_score: 0.95
action_required: False
action_requests:
action_responses:
```

### ReAct Final Answer:

```yaml
title: The Current State of Open-Source AI Agent Frameworks in 2025
summary: In 2025, open-source AI agent frameworks have matured significantly, with LangChain and LangGraph leading in popularity due to their flexibility, enterprise adoption, and robust feature sets. Emerging trends include modular multi-agent architectures, support for both visual and code-first development, enhanced memory and state management, and strong emphasis on deployment options including cloud and self-hosted environments. Frameworks like AutoGen, CrewAI, and n8n extend capabilities into automation workflows and multi-agent coordination, reflecting diverse use cases from rapid prototyping to production operations.
key_findings:
  - LangChain and LangGraph are the most popular open-source AI agent frameworks in 2025, with LangGraph boasting 34.5 million monthly downloads and LangChain widely recognized for its flexible API and ecosystem integration.
  - Frameworks like AutoGen and CrewAI are gaining traction for enabling advanced multi-agent collaboration and task delegation workflows.
  - The general-purpose automation platform n8n integrates AI nodes to build agentic workflows with strong orchestrator capabilities and extensive SaaS connectors.
  - A major trend is the support for multi-agent architectures allowing team orchestration, agent-to-agent communication, and human-in-the-loop workflows.
  - Frameworks increasingly emphasize modular orchestration models (graphs, workflows, function calls), memory/state management (episodic, session, long-term), and integrated tooling (search, code execution, vector databases).
  - Users balance trade-offs between visual builders (e.g., Langflow) for rapid prototyping and code-first APIs (e.g., AutoGen) for customization and production readiness.
  - Deployment flexibility is key, with many frameworks supporting cloud-hosted as well as self-hosted options to address privacy, governance, and scalability needs.
  - Enterprise adoption drives focus on governance features such as Role-Based Access Control (RBAC), audit logging, and integration with observability and security stacks.
  - Open model support continues to grow, with frameworks facilitating the use of various LLM providers including OpenAI, Anthropic, HuggingFace, and local models like Ollama.
  - The market for AI agents is expanding rapidly, with projections estimating over $7.8 billion in 2025, growing to over $52 billion by 2030, reflecting high demand for AI-driven automation and task-specific agents.
sources:
  - title: The Complete Guide to Choosing an AI Agent Framework in 2025
    url: https://www.langflow.org/blog/the-complete-guide-to-choosing-an-ai-agent-framework-in-2025
  - title: The Best Open Source Frameworks For Building AI Agents in 2026
    url: https://www.firecrawl.dev/blog/best-open-source-agent-frameworks
  - title: Top 10 Open-Source AI Agent Frameworks to Know in 2025 - ODSC
    url: https://odsc.medium.com/top-10-open-source-ai-agent-frameworks-to-know-in-2025-c739854ec859
  - title: Top AI Agent Frameworks in 2025 - Codecademy
    url: https://www.codecademy.com/article/top-ai-agent-frameworks-in-2025
```


Title: The Current State of Open-Source AI Agent Frameworks in 2025

Summary: In 2025, open-source AI agent frameworks have matured significantly, with LangChain and LangGraph leading in popularity due to their flexibility, enterprise adoption, and robust feature sets. Emerging trends include modular multi-agent architectures, support for both visual and code-first development, enhanced memory and state management, and strong emphasis on deployment options including cloud and self-hosted environments. Frameworks like AutoGen, CrewAI, and n8n extend capabilities into automation workflows and multi-agent coordination, reflecting diverse use cases from rapid prototyping to production operations.

Key Findings:
  - LangChain and LangGraph are the most popular open-source AI agent frameworks in 2025, with LangGraph boasting 34.5 million monthly downloads and LangChain widely recognized for its flexible API and ecosystem integration.
  - Frameworks like AutoGen and CrewAI are gaining tracti

## 5. Inspect the Conversation

See every step the agent took — reasoning, tool calls, and observations.

In [7]:
df = agent2.to_df()
print(f"Total conversation steps: {len(df)}")
print()

for _, row in df.iterrows():
    role = str(row["role"]).lower()
    content_str = str(row["content"])[:120]
    if "system" in role:
        print(f"  SYSTEM: (system message)")
    elif "user" in role:
        print(f"  USER: {content_str}...")
    elif "assistant" in role:
        print(f"  ASSISTANT: {content_str}...")
    elif "action" in role:
        print(f"  TOOL: {content_str}...")

Total conversation steps: 19

  SYSTEM: (system message)
  USER: {'images': [], 'instruction': 'Research the current state of open-source AI agent frameworks in 2025. Which are the most...
  ASSISTANT: {'assistant_response': '```json\n{"reason":{"title":"Initial planning and overview","content":"To produce a rigorous res...
  USER: {'images': [], 'instruction': 'You can perform multiple reason-action steps for accuracy. If you are not ready to finali...
  ASSISTANT: {'assistant_response': '```json\n{\n  "reason": {\n    "title": "Search for recent reviews and rankings of open-source A...
  USER: {'images': [], 'instruction': 'Another round is available. You may do multiple actions if needed. You have up to 7 expan...
  ASSISTANT: {'assistant_response': '```json\n{\n  "reason": {\n    "title": "Perform a web search for most popular open-source AI ag...
  TOOL: {'arguments': {'query': 'best open source AI agent frameworks 2025', 'max_results': 7, 'topic': 'general'}, 'action_resp...
  TO

## 6. Using Tavily & Firecrawl Directly (Without ReAct)

You can also use the endpoints directly for programmatic access.

In [8]:
from lionagi.service.connections.match_endpoint import match_endpoint

# Tavily search
tavily = match_endpoint("tavily", "search")
results = await tavily.call({"query": "lionagi framework", "max_results": 3, "include_answer": True})
print("Tavily answer:", results.get("answer"))
print(f"Got {len(results['results'])} results\n")

# Firecrawl scrape
firecrawl = match_endpoint("firecrawl", "scrape")
page = await firecrawl.call({"url": results["results"][0]["url"], "onlyMainContent": True})
if page.get("success"):
    print("Scraped page preview:")
    print(page["data"]["markdown"][:500])

# Firecrawl map
mapper = match_endpoint("firecrawl", "map")
sitemap = await mapper.call({"url": "https://github.com/lion-agi/lionagi", "limit": 10})
print("\nMapped URLs:")
for link in sitemap.get("links", [])[:5]:
    print(f"  {link}")

Tavily answer: LionAGI is an advanced intelligent agent framework that automates workflows and integrates machine learning models into existing systems. It is available on PyPI and YouTube for installation and demonstration.
Got 3 results



Scraped page preview:
Error 403 (Forbidden)!!1

**403.** That’s an error.

We're sorry, but you do not have access to this page. That’s all we know.

Back


[![](https://www.youtube.com/watch?v=ia9KRQiZCSM)](https://www.youtube.com/ "YouTube Home")

Skip navigation

Search


Search with your voice

[Sign in](https://accounts.google.com/ServiceLogin?service=youtube&uilel=3&passive=true&continue=https%3A%2F%2Fwww.youtube.com%2Fsignin%3Faction_handle_signin%3Dtrue%26app%3Ddesktop%26hl%3Den%26next%3Dhttps%253A%252F%252Fw



Mapped URLs:
  https://github.com/lion-agi/lionagi
  https://github.com/lion-agi/lionagi/issues
  https://github.com/lion-agi/lionagi/issues/247
  https://github.com/lion-agi/lionagi/issues/196
  https://github.com/lion-agi/lionagi/issues/394
